In [2]:
import pandas as pd
import numpy as np
import pickle
symbol = 'S'
df = pd.read_pickle(f'C:/Users/yuhang.hou/projects/holidays/poc_backtester/data_pipeline/universal/pickle_files/{symbol}_price.pickle')
df = df[df['type']=='close']
df


,date,contract,expiry_month,expiry_year,type,value
0,2025-01-01,S,8,2025,close,1032.75
3,2025-01-02,S,8,2025,close,1036.25
5,2025-01-03,S,8,2025,close,1015.50
7,2025-01-04,S,8,2025,close,1015.50
9,2025-01-05,S,8,2025,close,1015.50
...,...,...,...,...,...,...
241240,2014-05-14,S,5,2015,close,1237.00
241243,2014-05-15,S,5,2015,close,1230.00
241246,2014-05-16,S,5,2015,close,1233.75
241249,2014-05-19,S,5,2015,close,1249.25


In [3]:
business_days = list(df['date'].unique())
business_days = sorted(business_days)
business_days = [i.strftime('%Y-%m-%d') for i in business_days]


In [4]:
month_map = {1:'F',2:'G',3:'H',4:'J',5:'K',6:'M',7:'N',8:'Q',9:'U',10:'V',11:'X',12:'Z'}
df['contract_code'] = df['expiry_month'].map(month_map) + df['expiry_year'].astype(str).str[-2:]
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)
df

,contract,expiry_month,expiry_year,type,value,contract_code
date,,,,,,
2025-01-01,S,8,2025,close,1032.75,Q25
2025-01-02,S,8,2025,close,1036.25,Q25
2025-01-03,S,8,2025,close,1015.50,Q25
2025-01-04,S,8,2025,close,1015.50,Q25
2025-01-05,S,8,2025,close,1015.50,Q25
...,...,...,...,...,...,...
2014-05-14,S,5,2015,close,1237.00,K15
2014-05-15,S,5,2015,close,1230.00,K15
2014-05-16,S,5,2015,close,1233.75,K15


In [5]:
import os
rootpath =  f'C:/Users/yuhang.hou/projects/pipeline/data/{symbol}'
if not os.path.exists(rootpath):
    os.mkdir(rootpath)
contracts = df['contract_code'].unique()
ltd = {}
for contract in contracts:
    df_contract = df[df['contract_code'] == contract].copy()
    df_contract.sort_index( inplace=True)
    df_contract = df_contract[['value']].rename(columns={'value': 'close'})
    df_contract['return'] = df_contract['close'] / df_contract['close'].shift(1) - 1
    df_contract.to_csv(rootpath + '/' + contract + '.csv')
    if contract[1:]>'26':
        continue
    ltd[contract] = df_contract.index[-1].strftime('%Y-%m-%d')
    
ltd

{'Q25': '2025-07-31',
 'F26': '2026-01-19',
 'Q26': '2026-02-10',
 'H25': '2025-02-28',
 'U25': '2025-08-29',
 'H26': '2026-02-10',
 'K25': '2025-04-30',
 'K26': '2026-02-10',
 'N25': '2025-06-30',
 'N26': '2026-02-10',
 'U26': '2026-02-10',
 'X25': '2025-10-31',
 'X26': '2026-02-10',
 'X01': '2001-10-31',
 'F00': '1999-12-30',
 'U03': '2003-08-29',
 'Q05': '2005-07-29',
 'U05': '2005-08-31',
 'X03': '2003-10-31',
 'F09': '2008-12-31',
 'N07': '2007-06-29',
 'N10': '2010-06-30',
 'Q10': '2010-07-30',
 'H14': '2014-02-28',
 'F13': '2012-12-31',
 'X11': '2011-10-31',
 'N16': '2016-06-30',
 'K15': '2015-04-30',
 'N18': '2018-06-29',
 'K17': '2017-04-28',
 'Q19': '2019-07-31',
 'U20': '2020-08-31',
 'X21': '2021-10-29',
 'F23': '2022-12-30',
 'H24': '2024-02-29',
 'H00': '2000-02-29',
 'N15': '2015-06-30',
 'H13': '2013-02-28',
 'H09': '2009-02-27',
 'X20': '2020-10-30',
 'F02': '2001-12-31',
 'U19': '2019-08-30',
 'Q07': '2007-07-31',
 'H23': '2023-02-28',
 'X05': '2005-10-31',
 'U10': '2

In [7]:
import json
with open('last_trading_day.json', 'w') as f:
    json.dump(ltd, f)